In [ ]:
# Google Colab entry cell: separate Phase 7 ReInterHand baseline.
# It does not use the historical 5,114-image data or checkpoint.
# Keep START_TRAINING=False for preflight. Training starts only when changed to True.
from pathlib import Path
import json, os, runpy, sys, zipfile
from google.colab import drive, files

START_TRAINING = True
PHASE = "capture_holdout"
EPOCHS = 20
PAUSE_AT_MILESTONE = True
MILESTONE_GIB = 10.0
DRIVE_PROJECT = Path("/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker/04_DATASET_STAGING/ReInterHand")
TRAINING_READY = DRIVE_PROJECT / "03_training_ready"
EXTRACTED = DRIVE_PROJECT / "01_extracted"
BUNDLE_PATH = TRAINING_READY / "phase7_colab_bundle.zip"
CHECKPOINT_ROOT = TRAINING_READY / "checkpoints"
DEPLOYMENT_CHECKPOINT = CHECKPOINT_ROOT / "update_model.pt"
CONTROL_FILE = CHECKPOINT_ROOT / "phase7_command.txt"
EXPECTED = {"BKS383": 63100, "HIR112": 8581, "202107": 100815}
EXPECTED_TOTAL = sum(EXPECTED.values())

drive.mount("/content/drive")
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
if not CONTROL_FILE.is_file():
  CONTROL_FILE.write_text("continue", encoding="utf-8")
print(f"Drive control file: {CONTROL_FILE}")

report_path = TRAINING_READY / "FINAL_QUALITY_REPORT.json"
if not report_path.is_file():
    raise FileNotFoundError(f"Missing audit report: {report_path}")
report = json.loads(report_path.read_text(encoding="utf-8"))
if int(report.get("MANO_trainable_images", -1)) != EXPECTED_TOTAL or int(report.get("verified_images", -1)) != EXPECTED_TOTAL:
    raise RuntimeError("Audit totals are incomplete or differ from the prepared dataset")
report_captures = {item["label"]: item for item in report.get("captures", [])}
capture_roots = {}
for label, expected_count in EXPECTED.items():
    item = report_captures.get(label)
    if not item or int(item.get("images_trainable_MANO", -1)) != expected_count:
        raise RuntimeError(f"Unexpected audit count for {label}")
    manifest = TRAINING_READY / item["capture"] / "trainable_manifest.jsonl"
    overlay = TRAINING_READY / item["capture"] / "projection_overlay.png"
    if not manifest.is_file() or not overlay.is_file() or overlay.stat().st_size == 0:
        raise FileNotFoundError(f"Missing manifest or projection overlay for {label}")
    n = 0
    with manifest.open("r", encoding="utf-8") as stream:
        for line_no, line in enumerate(stream, 1):
            if not line.strip():
                continue
            row = json.loads(line)
            if not row.get("usable") or not row.get("mano_right") or not row.get("kp_right"):
                raise RuntimeError(f"Invalid target at {manifest}:{line_no}")
            n += 1
    if n != expected_count:
        raise RuntimeError(f"{label}: found {n:,} rows, expected {expected_count:,}")
    capture_name = item["capture"]
    candidates = [p for p in EXTRACTED.iterdir() if p.is_dir() and (p.name == capture_name or label.lower() in p.name.lower())]
    candidates = [p for p in candidates if
        (p / "frame_list.txt").is_file()
        and (p / "Mugsy_cameras" / "cam_params.json").is_file()
        and (p / "orig_fits" / "right" / "Keypoints").is_dir()
        and (p / "mano_fits" / "params").is_dir()
        and (p / "Mugsy_cameras" / "envmap_per_segment" / "images").is_dir()]
    if len(candidates) != 1:
        raise RuntimeError(f"Expected one extracted capture root for {label}; got {candidates}")
    capture_roots[label] = candidates[0]

print(f"Preflight passed: {EXPECTED_TOTAL:,} audited image/label joins")
print("Counts:", EXPECTED)
print("Initialization: fresh weights; historical 5,114-image data/checkpoint excluded.")
print("Recovery folder:", CHECKPOINT_ROOT)
print("Local-test endpoint:", DEPLOYMENT_CHECKPOINT)
if not BUNDLE_PATH.is_file():
    print("Upload the prepared phase7_colab_bundle.zip")
    uploaded = files.upload()
    zips = [(name, data) for name, data in uploaded.items() if name.lower().endswith(".zip")]
    if len(zips) != 1 or "phase7_colab_bundle" not in zips[0][0].lower():
        raise RuntimeError("Upload exactly phase7_colab_bundle.zip")
    BUNDLE_PATH.parent.mkdir(parents=True, exist_ok=True)
    BUNDLE_PATH.write_bytes(zips[0][1])

bundle_root = Path("/content/phase7_hm_ex_progress_v2")
trainer_path = bundle_root / "experiments" / "phase5_train_temporal.py"
if not trainer_path.is_file():
    bundle_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(BUNDLE_PATH) as archive:
        for member in archive.infolist():
            if not (bundle_root / member.filename).resolve().is_relative_to(bundle_root.resolve()):
                raise RuntimeError(f"Unsafe bundle path: {member.filename}")
        archive.extractall(bundle_root)
if not trainer_path.is_file():
    raise FileNotFoundError(f"Bundle is missing trainer: {trainer_path}")
if PHASE not in ("capture_holdout", "all_data") or EPOCHS < 1:
    raise ValueError("Invalid PHASE/EPOCHS")
if PHASE == "capture_holdout":
    checkpoint_dir = CHECKPOINT_ROOT / "phase7_capture_holdout"
    split_args = ["--split-mode", "capture", "--holdout-capture-token", "HIR112"]
    warmstart_args = []
else:
    first_best = CHECKPOINT_ROOT / "phase7_capture_holdout" / "best.pt"
    if not first_best.is_file():
        raise FileNotFoundError("Run and review capture_holdout before all_data")
    checkpoint_dir = CHECKPOINT_ROOT / "phase7_all_data"
    split_args = ["--split-mode", "segment", "--holdout-capture-token", ""]
    warmstart_args = ["--warmstart-checkpoint", str(first_best)]
argv = [str(trainer_path)] + [part for root in (json.loads((CHECKPOINT_ROOT / "all_indexed_capture_roots.json").read_text(encoding="utf-8"))["capture_roots"] if (CHECKPOINT_ROOT / "all_indexed_capture_roots.json").is_file() else capture_roots.values()) for part in ("--capture", str(root))] + [

    "--checkpoint-dir", str(checkpoint_dir), "--deployment-checkpoint", str(DEPLOYMENT_CHECKPOINT), "--control-file", str(CONTROL_FILE),
    "--device", "auto", "--epochs", str(EPOCHS), "--window-size", "6", "--batch-windows", "4", "--stride", "6",
    "--max-frame-gap", "30", "--image-size", "128", "--hidden-dim", "256", "--learning-rate", "0.0002",
    "--weight-decay", "0.0001", "--checkpoint-every-steps", "25", "--milestone-gib", str(MILESTONE_GIB), "--val-windows", "32",
    "--resume", *split_args, *warmstart_args]
if PAUSE_AT_MILESTONE:
    argv.append("--pause-at-milestone")

print(f"Phase={PHASE}; checkpoint directory={checkpoint_dir}")
if not START_TRAINING:
    print("PRE-FLIGHT ONLY. No training is running. Set START_TRAINING=True when ready.")
else:
    command = CONTROL_FILE.read_text(encoding="utf-8").strip().lower()
    if command in ("pause", "paused"):
        answer = input("Recovery checkpoint found. Type continue to resume from latest.pt: ").strip().lower()
        if answer != "continue":
            raise RuntimeError("Left paused; no training started.")
        CONTROL_FILE.write_text("continue", encoding="utf-8")
    os.environ["PYTHONPATH"] = str(bundle_root) + os.pathsep + os.environ.get("PYTHONPATH", "")
    sys.argv = argv
    import asyncio, threading, traceback
    import ipywidgets as widgets
    from IPython.display import display
    command_box = widgets.Text(value="", placeholder="pause", description="Command:", layout=widgets.Layout(width="330px"))
    send_button = widgets.Button(description="Send pause", button_style="warning")
    status = widgets.HTML(value="Type <b>pause</b> and click Send. Wait for <b>PAUSE SAVED</b> before disconnecting.")
    def send_pause(_):
        if command_box.value.strip().lower() != "pause":
            status.value = "Type <b>pause</b> in the box, then click Send."
            return
        tmp = CONTROL_FILE.with_suffix(CONTROL_FILE.suffix + ".tmp")
        tmp.write_text("pause", encoding="utf-8")
        os.replace(tmp, CONTROL_FILE)
        command_box.value = ""
        status.value = "Pause requested; saving after the current optimizer batch..."
    send_button.on_click(send_pause)
    display(widgets.HBox([command_box, send_button]), status)
    errors = []
    def run_training():
        try:
            runpy.run_path(str(trainer_path), run_name="__main__")
        except BaseException:
            errors.append(traceback.format_exc())
    worker = threading.Thread(target=run_training, name="phase7-training", daemon=True)
    worker.start()
    print("Training active. Type pause above, click Send, and wait for PAUSE SAVED.")
    while worker.is_alive():
        await asyncio.sleep(0.5)
    if errors:
        raise RuntimeError(errors[0])
    print("Trainer ended. For resume, rerun this cell with START_TRAINING=True and type continue at the prompt.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive control file: /content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker/04_DATASET_STAGING/ReInterHand/03_training_ready/checkpoints/phase7_command.txt
Preflight passed: 172,496 audited image/label joins
Counts: {'BKS383': 63100, 'HIR112': 8581, '202107': 100815}
Initialization: fresh weights; historical 5,114-image data/checkpoint excluded.
Recovery folder: /content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker/04_DATASET_STAGING/ReInterHand/03_training_ready/checkpoints
Local-test endpoint: /content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker/04_DATASET_STAGING/ReInterHand/03_training_ready/checkpoints/update_model.pt
Phase=capture_holdout; checkpoint directory=/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker/04_DATASET_STAGING/ReInterHand/03_training_ready/checkpoints/phase7_capture_holdout


HTML(value='Type <b>pause</b> and click Send. Wait for <b>PAUSE SAVED</b> before disconnecting.')

Training active. Type pause above, click Send, and wait for PAUSE SAVED.
Device: cuda
[index] Reused 100,815 cached records for m--20210701--1058--0000000--pilot--relightablehandsy--participant0--two-hands.
[index] Reused 63,100 cached records for m--20220628--1327--BKS383--pilot--ProjectGoliath--ContinuousHandsy--two-hands.
[index] Reused 7,765 cached records for m--20221007--1215--HIR112--pilot--ProjectGoliathScript--Hands--two-hands.
[index] Complete: 171,680 unique records; cached sources reused=3; elapsed 6:21.
[dataset] records=171,680; captures=3; split=capture
RAM: available=10.2 GiB; trainer=1.02 GiB
VRAM: free=14.5 / 14.6 GiB; device=Tesla T4
Checkpoint volume: free=60.9 GiB
Memory policy: RGB frames and MANO renders are loaded per sequence batch; the image corpus is never copied into RAM.
[targets] Loading/building camera labels and fitted HandState targets at /content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker/04_DATASET_STAGING/ReInterHand/03_training_ready/check

CancelledError: 

In [2]:
from pathlib import Path
import runpy
from google.colab import drive
drive.mount("/content/drive")
cell_path = Path("/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker/04_DATASET_STAGING/ReInterHand/03_training_ready/phase7_cpu_precompute_cell.py")
if not cell_path.is_file():
    raise FileNotFoundError(cell_path)
print("Running CPU-only resumable preprocessing; training will not start.")
runpy.run_path(str(cell_path), run_name="__main__")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Running CPU-only resumable preprocessing; training will not start.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Installed Drive-safe shard precomputer (sequential local image staging).
CPU preparation only; model training will not start.
Capture roots: ['/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker/04_DATASET_STAGING/ReInterHand/01_extracted/m--20210701--1058--0000000--pilot--relightablehandsy--participant0--two-hands', '/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker/04_DATASET_STAGING/ReInterHand/01_extracted/m--20220628--1327--BKS383--pilot--ProjectGoliath--ContinuousHandsy--two-hands', '/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker/04_DATASET_STAGING/ReInterHand/01_extracted/m--20221007--1215--HIR112--pilot--ProjectGoliathScri

{'__name__': '__main__',
 '__doc__': None,
 '__package__': '',
 '__loader__': None,
 '__spec__': None,
 '__file__': '/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker/04_DATASET_STAGING/ReInterHand/03_training_ready/phase7_cpu_precompute_cell.py',
 '__cached__': None,
 '__builtins__': {'__name__': 'builtins',
  '__doc__': "Built-in functions, types, exceptions, and other objects.\n\nThis module provides direct access to all 'built-in'\nidentifiers of Python; for example, builtins.len is\nthe full name for the built-in function len().\n\nThis module is not normally accessed explicitly by most\napplications, but can be useful in modules that provide\nobjects with the same name as a built-in value, but in\nwhich the built-in of that name is also needed.",
  '__package__': '',
  '__loader__': _frozen_importlib.BuiltinImporter,
  '__spec__': ModuleSpec(name='builtins', loader=<class '_frozen_importlib.BuiltinImporter'>, origin='built-in'),
  '__build_class__': <function __build_

In [1]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
print("Drive remounted; CPU runtime confirmed; cache and training cells remain stopped.")

Mounted at /content/drive
Drive remounted; CPU runtime confirmed; cache and training cells remain stopped.


In [ ]:
from pathlib import Path
import zipfile, py_compile
bundle = Path('/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker/04_DATASET_STAGING/ReInterHand/03_training_ready/phase7_colab_bundle.zip')
target = Path('/content/phase7_hm_ex_progress_v2/experiments/phase5_train_temporal.py')
with zipfile.ZipFile(bundle) as archive:
    target.write_bytes(archive.read('experiments/phase5_train_temporal.py'))
py_compile.compile(str(target), doraise=True)
source = target.read_text()
assert 'ThreadPoolExecutor' in source and 'args.batch_windows = 32' in source
print('Installed optimized trainer:', target)
from pathlib import Path
control = Path('/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker/04_DATASET_STAGING/ReInterHand/03_training_ready/checkpoints/phase7_command.txt')
control.write_text('continue\n')
print('Training control:', control.read_text().strip())

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker/04_DATASET_STAGING/ReInterHand/03_training_ready/phase7_colab_bundle.zip'

In [3]:
# Cached-input Phase 7 trainer launcher (GPU runtime required).
from pathlib import Path
from IPython import get_ipython
from google.colab import drive

drive.mount("/content/drive")
runner_path = Path("/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker/04_DATASET_STAGING/ReInterHand/03_training_ready/phase7_vram_training_cell.py")
if not runner_path.is_file():
    raise FileNotFoundError(f"Upload phase7_vram_training_cell.py to Drive first: {runner_path}")
runner_source = runner_path.read_text(encoding="utf-8")
assert 'phase7_vram_bundle' in runner_source and '--precomputed-cache' in runner_source
print("Launching verified cached-input training cell from Drive:", runner_path, flush=True)
cell_result = await get_ipython().run_cell_async(runner_source, store_history=True)
if cell_result.error_before_exec or cell_result.error_in_exec:
    raise RuntimeError("Cached-input training cell failed; inspect the output above.")


Mounted at /content/drive
Launching verified cached-input training cell from Drive: /content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker/04_DATASET_STAGING/ReInterHand/03_training_ready/phase7_vram_training_cell.py


/tmp/ipykernel_370/242956449.py:13: DeprecationWarning: `run_cell_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  cell_result = await get_ipython().run_cell_async(runner_source, store_history=True)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Live training control file: /content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker/04_DATASET_STAGING/ReInterHand/03_training_ready/checkpoints/phase7_command.txt (the running cell's Command box writes pause/continue)
Audit preflight passed: 172,496 verified image/label joins
Images by capture: {'BKS383': 63100, 'HIR112': 8581, '202107': 100815}
Data roots: {'BKS383': '/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker/04_DATASET_STAGING/ReInterHand/01_extracted/m--20220628--1327--BKS383--pilot--ProjectGoliath--ContinuousHandsy--two-hands', 'HIR112': '/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker/04_DATASET_STAGING/ReInterHand/01_extracted/m--20221007--1215--HIR112--pilot--ProjectGoliathScript--Hands--two-hands', '202107': '/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker/04_DATASET_STAGING/ReInterHand/01_extract

HTML(value='Type <b>pause</b> and click Send. Wait for the trainer to print PAUSE SAVED before disconnecting.'…

Training started. Use the Command box above: type pause, click Send, and wait for PAUSE SAVED.
Device: cuda
[index] Scanning capture m--20220628--1327--BKS383--pilot--ProjectGoliath--ContinuousHandsy--two-hands; progress is reported every 2,048 candidates.
[index] m--20220628--1327--BKS383--pilot--ProjectGoliath--ContinuousHandsy--two-hands: 1/63,100 candidates (0.0%), camera 400299; elapsed 0:00, ETA 7:43:03


CancelledError: 

RuntimeError: Cached-input training cell failed; inspect the output above.

In [ ]:
from pathlib import Path
import torch
from google.colab import drive

drive.mount("/content/drive")
runner_path = Path("/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker/04_DATASET_STAGING/ReInterHand/03_training_ready/phase7_vram_training_cell.py")
if not runner_path.is_file():
    raise FileNotFoundError(runner_path)
    runner_source = runner_path.read_text(encoding="utf-8")
    required = [
        'INDEX_CACHE_DIR = CHECKPOINT_ROOT / "phase7_capture_holdout" / "record_index"',
            '"--precomputed-cache", str(PRECOMPUTED_CACHE)',
                '"--ram-cache-shards", "12"',
                    '"--checkpoint-every-steps", "10"',
                        '"--webcam-checkpoint-every-epochs", "5"',
                            'if resume_existing_run:',
                                'argv.append("--resume")',
                                    'CHECKPOINT_ROOT / "phase7_capture_holdout_cached"',
                                    ]
                                    missing = [item for item in required if item not in runner_source]
                                    if missing:
                                        raise RuntimeError("Drive runner does not match the fresh cached-input plan: " + repr(missing))
                                        if not torch.cuda.is_available():
                                            raise RuntimeError("CUDA is unavailable. Select a GPU runtime before starting training.")
                                            print("Verified fresh-start runner, cached input bank, reusable capture indexes, 12-shard RAM prefetch, and checkpoint cadence.", flush=True)
                                            print("GPU:", torch.cuda.get_device_name(0), flush=True)
                                            cell_result = await get_ipython().run_cell_async(runner_source, store_history=True)
                                            if cell_result.error_before_exec or cell_result.error_in_exec:
                                                raise RuntimeError("Fresh training runner failed; see the traceback above.")
                                                from pathlib import Path
import torch
from google.colab import drive

drive.mount("/content/drive")
runner_path = Path("/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker/04_DATASET_STAGING/ReInterHand/03_training_ready/phase7_vram_training_cell.py")
if not runner_path.is_file():
    raise FileNotFoundError(runner_path)
runner_source = runner_path.read_text(encoding="utf-8")
required = [
    'INDEX_CACHE_DIR = CHECKPOINT_ROOT / "phase7_capture_holdout" / "record_index"',
    '"--precomputed-cache", str(PRECOMPUTED_CACHE)',
    '"--ram-cache-shards", "12"',
    '"--checkpoint-every-steps", "10"',
    '"--webcam-checkpoint-every-epochs", "5"',
    'if resume_existing_run:
    argv.append("--resume")',
    'CHECKPOINT_ROOT / "phase7_capture_holdout_cached"',
]
missing = [item for item in required if item not in runner_source]
if missing:
    raise RuntimeError("Drive runner does not match the fresh cached-input plan: " + repr(missing))
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Select a GPU runtime before starting training.")
print("Verified fresh-start runner, cached input bank, reusable capture indexes, 12-shard RAM prefetch, and checkpoint cadence.", flush=True)
print("GPU:", torch.cuda.get_device_name(0), flush=True)
cell_result = await get_ipython().run_cell_async(runner_source, store_history=True)
if cell_result.error_before_exec or cell_result.error_in_exec:
    raise RuntimeError("Fresh training runner failed; see the traceback above.")


In [ ]:
from pathlib import Path
import torch
from google.colab import drive
drive.mount("/content/drive")
runner_path = Path("/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker/04_DATASET_STAGING/ReInterHand/03_training_ready/phase7_vram_training_cell.py")
assert runner_path.is_file(), str(runner_path)
runner_source = runner_path.read_text(encoding="utf-8")
assert all(s in runner_source for s in ('INDEX_CACHE_DIR = CHECKPOINT_ROOT / "phase7_capture_holdout" / "record_index"', '"--precomputed-cache", str(PRECOMPUTED_CACHE)', '"--ram-cache-shards", "12"', '"--checkpoint-every-steps", "10"', '"--webcam-checkpoint-every-epochs", "5"', 'if resume_existing_run:', 'argv.append("--resume")', 'CHECKPOINT_ROOT / "phase7_capture_holdout_cached"')), "Drive runner differs from the fresh cached-input plan"
assert torch.cuda.is_available(), "CUDA unavailable: select the GPU runtime before training"
print("Verified fresh-start cached-input runner on", torch.cuda.get_device_name(0), flush=True)
cell_result = await get_ipython().run_cell_async(runner_source, store_history=True)
assert cell_result.error_before_exec is None and cell_result.error_in_exec is None, "Fresh trainer failed; inspect the preceding output"
from pathlib import Path
import torch
from google.colab import drive

drive.mount("/content/drive")
runner_path = Path("/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker/04_DATASET_STAGING/ReInterHand/03_training_ready/phase7_vram_training_cell.py")
if not runner_path.is_file():
    raise FileNotFoundError(runner_path)
runner_source = runner_path.read_text(encoding="utf-8")
required = [
    'INDEX_CACHE_DIR = CHECKPOINT_ROOT / "phase7_capture_holdout" / "record_index"',
    '"--precomputed-cache", str(PRECOMPUTED_CACHE)',
    '"--ram-cache-shards", "12"',
    '"--checkpoint-every-steps", "10"',
    '"--webcam-checkpoint-every-epochs", "5"',
    'if resume_existing_run:',
    'argv.append("--resume")',
    'CHECKPOINT_ROOT / "phase7_capture_holdout_cached"',
]
missing = [item for item in required if item not in runner_source]
if missing:
    raise RuntimeError("Drive runner differs from fresh cached-input plan: " + repr(missing))
if not torch.cuda.is_available():
    raise RuntimeError("CUDA unavailable; select a GPU runtime before training")
print("Verified fresh-start cached-input runner on", torch.cuda.get_device_name(0), flush=True)
cell_result = await get_ipython().run_cell_async(runner_source, store_history=True)
if cell_result.error_before_exec or cell_result.error_in_exec:
    raise RuntimeError("Fresh trainer failed; inspect preceding output")

In [1]:
from pathlib import Path
import torch
from google.colab import drive

drive.mount("/content/drive")
runner_path = Path("/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker/04_DATASET_STAGING/ReInterHand/03_training_ready/phase7_vram_training_cell.py")
if not runner_path.is_file():
    raise FileNotFoundError(runner_path)
runner_source = runner_path.read_text(encoding="utf-8")
required = [
    'INDEX_CACHE_DIR = CHECKPOINT_ROOT / "phase7_capture_holdout" / "record_index"',
    '"--precomputed-cache", str(PRECOMPUTED_CACHE)',
    '"--ram-cache-shards", "12"',
    '"--checkpoint-every-steps", "10"',
    '"--webcam-checkpoint-every-epochs", "5"',
    'if resume_existing_run:',
    'argv.append("--resume")',
    'CHECKPOINT_ROOT / "phase7_capture_holdout_cached"',
]
missing = [item for item in required if item not in runner_source]
if missing:
    raise RuntimeError("Drive runner differs from fresh cached-input plan: " + repr(missing))
if not torch.cuda.is_available():
    raise RuntimeError("CUDA unavailable; select a GPU runtime before training")
print("Verified fresh-start cached-input runner on", torch.cuda.get_device_name(0), flush=True)
cell_result = await get_ipython().run_cell_async(runner_source, store_history=True)
if cell_result.error_before_exec or cell_result.error_in_exec:
    raise RuntimeError("Fresh trainer failed; inspect preceding output")

Mounted at /content/drive


RuntimeError: CUDA unavailable; select a GPU runtime before training

In [2]:
from pathlib import Path
from google.colab import drive, files
import shutil

drive.mount("/content/drive")

src = Path(
    "/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker/"
    "04_DATASET_STAGING/ReInterHand/03_training_ready/checkpoints/update_model.pt"
)

bundle_roots = [
    Path("/content/phase7_hm_ex_progress_v2"),
    Path("/content/phase7_vram_bundle"),
]
bundle = next(
    (p for p in bundle_roots if (p / "experiments/phase7_webcam_smoke.py").is_file()),
    None,
)

if not src.is_file():
    raise FileNotFoundError(f"Checkpoint not found in Drive: {src}")
if bundle is None:
    raise FileNotFoundError("Phase 7 bundle not found in /content.")

dest = bundle / "update_model.pt"
shutil.copy2(src, dest)
print(f"Copied {dest} ({dest.stat().st_size / 1024**2:.1f} MiB)")
files.download(str(dest))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


FileNotFoundError: Phase 7 bundle not found in /content.

In [1]:
from pathlib import Path
from google.colab import drive, files
import shutil

drive.mount("/content/drive")

src = Path(
    "/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker/"
    "04_DATASET_STAGING/ReInterHand/03_training_ready/checkpoints/update_model.pt"
)

bundle_roots = [
    Path("/content/phase7_hm_ex_progress_v2"),
    Path("/content/phase7_vram_bundle"),
]
bundle = next(
    (p for p in bundle_roots if (p / "experiments/phase7_webcam_smoke.py").is_file()),
    None,
)

if not src.is_file():
    raise FileNotFoundError(f"Checkpoint not found in Drive: {src}")
if bundle is None:
    raise FileNotFoundError("Phase 7 bundle not found in /content.")

dest = bundle / "update_model.pt"
shutil.copy2(src, dest)
print(f"Copied {dest} ({dest.stat().st_size / 1024**2:.1f} MiB)")
files.download(str(dest))

Mounted at /content/drive


FileNotFoundError: Phase 7 bundle not found in /content.

In [2]:
from pathlib import Path
import shutil
from google.colab import drive, files

drive.mount("/content/drive")

source = Path(
    "/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker/"
    "04_DATASET_STAGING/ReInterHand/03_training_ready/checkpoints/"
    "update_model.pt"
)
destination = Path("/content/update_model.pt")

if not source.is_file():
    raise FileNotFoundError(f"Checkpoint not found in Drive: {source}")

shutil.copyfile(source, destination)

source_size = source.stat().st_size
download_size = destination.stat().st_size
if source_size == 0 or source_size != download_size:
    raise IOError(
        f"Checkpoint copy verification failed: Drive={source_size} bytes, "
        f"Colab={download_size} bytes"
    )

print(f"Checkpoint copied and verified: {download_size / (1024**2):.1f} MiB")
print("Starting browser download as update_model.pt")
files.download(str(destination))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checkpoint copied and verified: 16.9 MiB
Starting browser download as update_model.pt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>